# Summary

Create text files from previously crawls.

In [21]:
import os, sys

import pandas as pd

utils_path = "../../interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication
import gcp_tools as gct

# Set environment variables
dotenv_path = "../../data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()

# Initialize Vertex AI API once per session
# vertexai.init(project=os.environ["GOOGLE_CLOUD_PROJECT"],
#               location=os.environ["GOOGLE_CLOUD_LOCATION"],
#               staging_bucket=os.environ["STAGING_BUCKET"])


In [22]:
gcs_bucket_name = "ccc-crawl_data_xb"
bcontents = gct.gcp_list_bucket(gcp_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                gcs_bucket_name=gcs_bucket_name,
                                path="")



In [25]:

cfiles = []
for bc in bcontents:
    fp = os.path.split(bc)
    ps = fp[0].split("/")

    source = ps[1] if len(ps) > 1 else ""
    file_name = fp[1] if fp[1].find(".csv") >= 0 else ""

    if len(file_name) > 0:
        cfiles.append(dict(source=source,
                           file_name=file_name,
                           path=fp[0]))

dfc = pd.DataFrame(data=cfiles)

dfc


    # if bc["name"].endswith(".csv"):
    #     print(bc["name"])
    #     gct.gcp_download_blob(gcp_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],)

,source,file_name,path
0,aacc,wwwaaccncheedu_2025May01_1.csv,crawl_data/aacc/webpages_pdfs
1,aacc,wwwaaccncheedu_2025May01_2.csv,crawl_data/aacc/webpages_pdfs
2,aacc,wwwaaccncheedu_2025May01_3.csv,crawl_data/aacc/webpages_pdfs
3,aacc,wwwaaccncheedu_2025May01_4.csv,crawl_data/aacc/webpages_pdfs
4,aacc,wwwaaccncheedu_2025May01_5.csv,crawl_data/aacc/webpages_pdfs
...,...,...,...
81,nsc,nscresearchcenterorg_2025May01_3.csv,crawl_data/nsc/webpages_pdfs
82,wikipedia,enwikipediaorg_2025May01_1.csv,crawl_data/wikipedia/webpages_pdfs
83,wikipedia,enwikipediaorg_2025May01_2.csv,crawl_data/wikipedia/webpages_pdfs
84,wikipedia,enwikipediaorg_2025May01_3.csv,crawl_data/wikipedia/webpages_pdfs


In [50]:

mask = dfc["source"] == "aacc"
mask = dfc["source"] == "wikipedia"
mask = dfc["source"] == "ipeds"


dfs = []
for idx in dfc[mask].index[:1]:
    print(dfc.loc[idx, "file_name"])
    df = gct.read_csv_file_into_pandas(gcs_project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
                                       gcs_bucket_name=gcs_bucket_name,
                                       gcs_directory=dfc.loc[idx, "path"],
                                       file_name=dfc.loc[idx, "file_name"],
                                       exact=False)
    dfs.append(df)

df = pd.concat(dfs)

df = df.reset_index(drop=True)




prep_2025Apr29_c2023_a.csv


In [51]:
df

,Unnamed: 0,UNITID,CIPCODE,MAJORNUM,AWLEVEL,XCTOTALT,CTOTALT,XCTOTALM,CTOTALM,XCTOTALW,...,CUNKNM,XCUNKNW,CUNKNW,XCNRALT,CNRALT,XCNRALM,CNRALM,XCNRALW,CNRALW,college_name
0,0,108667,5.0200,1,3,R,0,R,0,Z,...,0,Z,0,Z,0,Z,0,Z,0,COLLEGE OF ALAMEDA
1,1,108667,9.0101,1,3,R,2,R,2,Z,...,0,Z,0,Z,0,Z,0,Z,0,COLLEGE OF ALAMEDA
2,2,108667,11.0103,1,3,R,1,R,1,Z,...,0,Z,0,Z,0,Z,0,Z,0,COLLEGE OF ALAMEDA
3,3,108667,11.0103,1,20,R,0,R,0,Z,...,0,Z,0,Z,0,Z,0,Z,0,COLLEGE OF ALAMEDA
4,4,108667,11.0103,1,21,R,1,R,1,Z,...,0,Z,0,Z,0,Z,0,Z,0,COLLEGE OF ALAMEDA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14314,14314,489201,54.0101,1,4,R,0,R,0,Z,...,0,Z,0,R,0,R,0,Z,0,CLOVIS COMMUNITY COLLEGE
14315,14315,489201,99.0000,1,2,R,0,R,0,Z,...,0,Z,0,R,0,R,0,Z,0,CLOVIS COMMUNITY COLLEGE
14316,14316,489201,99.0000,1,3,R,1441,R,529,R,...,2,R,10,R,0,R,0,Z,0,CLOVIS COMMUNITY COLLEGE
14317,14317,489201,99.0000,1,4,R,0,R,0,Z,...,0,Z,0,R,0,R,0,Z,0,CLOVIS COMMUNITY COLLEGE


In [42]:
df.loc[2, "ptag_text"]

'Orange Coast College (OCC) is a public community college in Costa Mesa in Orange County, California. It was founded in 1947, with its first classes opening in the fall of 1948. It provides Associate of Art and Associate of Science degrees, certificates of achievement, and lower-division classes transferable to other colleges and universities. The college enrolls approximately 24,000 undergraduate students. In terms of population size, Orange Coast College is the third-largest college in Orange County. Orange Coast College was formed after local voters passed a measure in the January 1947 election to establish a new junior college on a 243-acre (0.98 km2) site, secured from the War Assets Administration in Washington, D.C., and part of the 1,300-acre (5.3 km2) deactivated Santa Ana Army Air Base. The first official District board of trustees hired the college\'s founding president and district superintendent, Basil Hyrum Peterson, on July 28, 1947. Construction of campus classrooms and

In [43]:
txt_list = []
for idx in df.index:
    txt = "{}\n{}".format(df.loc[idx, "page_title"],
                          df.loc[idx, "ptag_text"])
    txt_list.append(txt)




In [38]:
txt = "\n\n".join(txt_list)

txt

r = "\[…]"

'Home Page - AACC\nSearch our Community College Finder Search upcoming AACC Events Search the AACC Member Directory Season 10, Episode 1: Talking to Pasco-Hernando State College about Cultural Change Through Listening, Learning and Acting AACC John E. Roueche Future Leaders Institute AACC Future Presidents Institute June 2-4, 2025 Washington, DC 2025 Presidents Academy Summer Institute July 12-15, 2025 The Ritz Carlton, Laguna Beach, CA The fourth edition of the Competencies is a fully comprehensive document to guide the development of employees dedicated to the community college mission, vision, and values. Buy now. ©2025 American Association of Community Colleges One Dupont Circle, NW, Suite 700 Washington, DC 20036\n\nResearch - AACC\nCommunity colleges are centers of educational opportunity. They are an American invention that put publicly funded higher education at close-to-home facilities, beginning nearly 100 years ago with Joliet Junior College. Since then, community colleges h

In [39]:
len(txt)

1349884